In [1]:
import sys
import os

# Get the absolute path to the parent directory (CSCI5527-FINAL)
parent_dir = os.path.abspath("..")

# Add the parent directory to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
import torch
import torch.optim as optim
import torch.nn as nn
import segmentation_models_pytorch as smp
from fastai.losses import DiceLoss, CrossEntropyLossFlat

import io
import pandas as pd
from contextlib import redirect_stdout, redirect_stderr
import tqdm

from matrices import *
from preprocessing import *
from utils import *

In [3]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.backends.cudnn.is_available())
n_gpu = torch.cuda.device_count()
print(f"Total GPUs available: {n_gpu}")
for i in range(n_gpu):
    print(f"Device {i}: {torch.cuda.get_device_name(i)}")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using {device} device")
generator = torch.Generator(device=device)

2.11.0+cu126
True
True
Total GPUs available: 1
Device 0: NVIDIA GeForce RTX 2060 SUPER
Using cuda:0 device


In [4]:
def baseline_model_pipeline_imagenet(base_dir, dict_files):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = None # use imagenet for baseline

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [5]:
config = {
    "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
    "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
    "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
}
train_dl_imagenet, val_dl_imagenet, test_dl_imagenet, custom_stats_imagenet = baseline_model_pipeline_imagenet(parent_dir, config)

Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 1360/1360 [00:00<00:00, 30107.80it/s]


Sanitizing test masks...


Sanitizing Masks: 100%|██████████| 156/156 [00:00<00:00, 31976.90it/s]

Generating Dataloaders...



Pipeline Ready:
 - Training batches: 66
 - Validation batches: 19
 - Testing batches: 10


In [6]:
def baseline_model_pipeline_no_weight(base_dir, dict_files):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = pipeline.calculate_training_stats(df_train_val_ready)

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [7]:
config = {
    "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
    "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
    "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
}
train_dl_no_weight, val_dl_no_weight, test_dl_no_weight, custom_stats_no_weight = baseline_model_pipeline_no_weight(parent_dir, config)

Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 1360/1360 [00:00<00:00, 30243.64it/s]


Sanitizing test masks...


Sanitizing Masks: 100%|██████████| 156/156 [00:00<00:00, 28096.51it/s]


Calculating custom dataset statistics...


Computing Stats: 100%|██████████| 1058/1058 [00:18<00:00, 56.69it/s]


Generating Dataloaders...

Pipeline Ready:
 - Training batches: 66
 - Validation batches: 19
 - Testing batches: 10


In [9]:
# because the preprocessing with the HPF is updated, so we need to re-import the functions in preprocessing_wi_filter.py
from preprocessing_wi_filter import *

In [10]:
def baseline_model_pipeline_sobelxy(base_dir, dict_files, processed_version="4_img_sobelxy"):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir,
        processed_img_subdir=processed_version
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = pipeline.calculate_training_stats(df_train_val_ready)

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [11]:
config = {
    "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
    "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
    "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
}
processed_version = "4_img_sobelxy"
train_dl_sobelxy, val_dl_sobelxy, test_dl_sobelxy, custom_stats_sobelxy = baseline_model_pipeline_sobelxy(parent_dir, config, processed_version=processed_version)

Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 1360/1360 [00:00<00:00, 24469.38it/s]


Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera8_000001_H_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera4_000186_A_D_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera5_000178_B_D_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera5_000058_M_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera4_000148_C_B_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera7_000125_B_B_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera

Sanitizing Masks: 100%|██████████| 156/156 [00:00<00:00, 21967.82it/s]


Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera5_000030_L_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera8_000058_D_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera8_000058_I_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera5_000173_D_C_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera7_000058_H_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera7_000174_A_C_sobelxy.png
Skipping: Processed image not found at /home/yash/Documents/GitHub/CSCI5527-final/TACK_Tunnel_Data/4_img_sobelxy/TA_Camera4_00

Computing Stats: 0it [00:00, ?it/s]

Generating Dataloaders...



/home/yash/Documents/GitHub/CSCI5527-final/.conda/lib/python3.11/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/yash/Documents/GitHub/CSCI5527-final/.conda/lib/python3.11/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


TypeError: 'NoneType' object is not iterable

In [ ]:
def baseline_model_pipeline_sobelmaglap(base_dir, dict_files, processed_version="4_img_sobelxy"):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir,
        processed_img_subdir=processed_version
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = pipeline.calculate_training_stats(df_train_val_ready)

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [ ]:
config = {
    "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
    "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
    "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
}
processed_version = "4_img_sobelmaglap"
train_dl_sobelmaglap, val_dl_sobelmaglap, test_dl_sobelmaglap, custom_stats_sobelmaglap = baseline_model_pipeline_sobelmaglap(parent_dir, config, processed_version=processed_version)

In [ ]:
def baseline_model_pipeline_sobelmag_gabor(base_dir, dict_files, processed_version="4_img_sobelxy"):
    # Locate project root and TACK data folders dynamically relative to the script location
    dataset_folder = os.path.join(base_dir, "TACK_Tunnel_Data")

    csv_source_dir = os.path.join(dataset_folder, "2_model_input")
    raw_mask_dir = os.path.join(dataset_folder, "3_mask")

    # Initialize pipeline pointing to the TTD dataset structure
    pipeline = TunnelDataPipeline(
        base_dir=dataset_folder,
        original_mask_dir=raw_mask_dir,
        processed_img_subdir=processed_version
    )

    # Define standard training splits for the multi-domain tunnel study
    train_files = dict_files["train_files"]
    val_files = dict_files["val_files"]
    test_files = dict_files["test_files"]

    # Step 1: Load metadata
    print("Loading CSV metadata...")
    df_train_val, df_test = pipeline.load_csv_data(
        csv_source_dir=csv_source_dir,
        train_files=train_files,
        val_files=val_files,
        test_files=test_files
    )

    # Step 2: Sanitize and separate training/validation masks
    print("Sanitizing training and validation masks...")
    df_train_val_ready = pipeline.sanitize_masks(df_train_val, class_pixel_value=40)

    # Step 3: Sanitize test masks
    print("Sanitizing test masks...")
    df_test_ready = pipeline.sanitize_masks(df_test, class_pixel_value=40)

    # Step 4: Compute dataset-specific normalization values
    custom_stats = pipeline.calculate_training_stats(df_train_val_ready)

    # Step 5: Finalize DataLoaders for the training loop
    print("Generating Dataloaders...")
    train_dl, val_dl, test_dl = pipeline.get_dataloaders(
        train_val_df=df_train_val_ready,
        test_df=df_test_ready,
        bs=16,
        img_size=512,
        custom_stats=custom_stats
    )

    # Final summary of training readiness
    print(f"\nPipeline Ready:")
    print(f" - Training batches: {len(train_dl)}")
    print(f" - Validation batches: {len(val_dl)}")
    print(f" - Testing batches: {len(test_dl)}")

    return train_dl, val_dl, test_dl, custom_stats

In [ ]:
config = {
    "train_files": ["TA_train.csv", "TB_train.csv", "TC_train.csv"],
    "val_files": ["TA_val.csv", "TB_val.csv", "TC_val.csv"],
    "test_files": ["TA_test.csv", "TB_test.csv", "TC_test.csv"]
}
processed_version = "4_img_sobelmag_gabor"
train_dl_sobelmag_gabor, val_dl_sobelmag_gabor, test_dl_sobelmag_gabor, custom_stats_sobelmag_gabor = baseline_model_pipeline_sobelmag_gabor(parent_dir, config, processed_version=processed_version)

In [ ]:
import math
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch.nn.functional as F

In [ ]:
def load_unet_resnet34_model(model_name, device):
    model = smp.Unet("resnet34", encoder_weights="imagenet", classes=2)
    checkpoint_path = Path("models") / f"{model_name}.pth"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {checkpoint_path}")

    checkpoint = torch.load(checkpoint_path, map_location=device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        checkpoint = checkpoint["model_state_dict"]

    model.load_state_dict(checkpoint)
    model.to(device)
    model.eval()
    return model


def get_resnet34_feature_layers(model):
    return {
        "low": model.encoder.layer1[-1],
        "mid": model.encoder.layer3[-1],
        "high": model.encoder.layer4[-1],
        "classifier": model.segmentation_head,
    }


model_name_imagenet = "Unet-resnet34-imagenet_Multi-Domain"
model_name_no_weight = "Unet-resnet34-no_weight_Multi-Domain"
model_name_imagenet_sobelxy = "Unet-resnet34-sobelxy_Multi-Domain"
model_name_imagenet_sobelmaglap = "Unet-resnet34-sobelmaglap_Multi-Domain"
model_name_imagenet_sobelmag_gabor = "Unet-resnet34-sobelmag_gabor_Multi-Domain"

model_imagenet = load_unet_resnet34_model(model_name_imagenet, device)
model_no_weight = load_unet_resnet34_model(model_name_no_weight, device)
model_imagenet_sobelxy = load_unet_resnet34_model(model_name_imagenet_sobelxy, device)
model_imagenet_sobelmaglap = load_unet_resnet34_model(model_name_imagenet_sobelmaglap, device)
model_imagenet_sobelmag_gabor = load_unet_resnet34_model(model_name_imagenet_sobelmag_gabor, device)

models_to_compare = [
    model_imagenet,
    model_no_weight,
    model_imagenet_sobelxy,
    model_imagenet_sobelmaglap,
    model_imagenet_sobelmag_gabor,
]

names_to_compare = [
    model_name_imagenet,
    model_name_no_weight,
    model_name_imagenet_sobelxy,
    model_name_imagenet_sobelmaglap,
    model_name_imagenet_sobelmag_gabor,
]

feature_layers_to_compare = [get_resnet34_feature_layers(model) for model in models_to_compare]

In [ ]:
fallback_dataloaders = [
    [test_dl_imagenet, val_dl_imagenet, train_dl_imagenet],
    [test_dl_no_weight, val_dl_no_weight, train_dl_no_weight],
    [test_dl_sobelxy, val_dl_sobelxy, train_dl_sobelxy],
    [test_dl_sobelmaglap, val_dl_sobelmaglap, train_dl_sobelmaglap],
    [test_dl_sobelmag_gabor, val_dl_sobelmag_gabor, train_dl_sobelmag_gabor],
]

custom_stats_list = [
    custom_stats_imagenet,
    custom_stats_no_weight,
    custom_stats_sobelxy,
    custom_stats_sobelmaglap,
    custom_stats_sobelmag_gabor,
]

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def unpack_batch(batch):
    if isinstance(batch, dict):
        image = None
        target = None
        for key in ["image", "images", "x", "input", "inputs"]:
            if key in batch:
                image = batch[key]
                break
        for key in ["mask", "masks", "target", "targets", "y", "label", "labels"]:
            if key in batch:
                target = batch[key]
                break
        if image is None:
            raise KeyError("Could not find an image tensor in the batch dictionary.")
        return image, target

    if isinstance(batch, (list, tuple)):
        image = batch[0]
        target = batch[1] if len(batch) > 1 else None
        return image, target

    return batch, None


def first_available_sample(dataloaders, device):
    last_error = None
    for dataloader in dataloaders:
        try:
            batch = next(iter(dataloader))
            image, target = unpack_batch(batch)
            if image.ndim == 3:
                image = image.unsqueeze(0)
            image = image[:1].to(device)
            if target is not None and hasattr(target, "to"):
                target = target[:1].to(device)
            return image, target
        except Exception as exc:
            last_error = exc
            print(f"Skipping dataloader because it could not provide a sample: {exc}")
    raise RuntimeError("No dataloader produced a valid sample.") from last_error


def _extract_mean_std(custom_stats, channels):
    if custom_stats is None:
        if channels == 3:
            return IMAGENET_MEAN, IMAGENET_STD
        return np.zeros(channels, dtype=np.float32), np.ones(channels, dtype=np.float32)

    if isinstance(custom_stats, dict):
        mean = custom_stats.get("mean", custom_stats.get("means", None))
        std = custom_stats.get("std", custom_stats.get("stds", None))
    elif isinstance(custom_stats, (list, tuple)) and len(custom_stats) == 2:
        mean, std = custom_stats
    else:
        mean, std = None, None

    if mean is None or std is None:
        if channels == 3:
            return IMAGENET_MEAN, IMAGENET_STD
        return np.zeros(channels, dtype=np.float32), np.ones(channels, dtype=np.float32)

    if torch.is_tensor(mean):
        mean = mean.detach().cpu().numpy()
    if torch.is_tensor(std):
        std = std.detach().cpu().numpy()

    mean = np.asarray(mean, dtype=np.float32).reshape(-1)
    std = np.asarray(std, dtype=np.float32).reshape(-1)

    if mean.size == 1 and channels > 1:
        mean = np.repeat(mean, channels)
    if std.size == 1 and channels > 1:
        std = np.repeat(std, channels)

    if mean.size != channels or std.size != channels:
        if channels == 3:
            return IMAGENET_MEAN, IMAGENET_STD
        return np.zeros(channels, dtype=np.float32), np.ones(channels, dtype=np.float32)

    return mean, std


def tensor_to_display_image(image_tensor, custom_stats=None):
    image = image_tensor.detach().cpu().float()
    if image.ndim == 4:
        image = image[0]

    channels = image.shape[0]
    mean, std = _extract_mean_std(custom_stats, channels)
    mean = torch.tensor(mean, dtype=image.dtype, device=image.device).view(-1, 1, 1)
    std = torch.tensor(std, dtype=image.dtype, device=image.device).view(-1, 1, 1)

    image = image * std + mean
    image = image.clamp(0, 1)

    if channels == 1:
        image = image.repeat(3, 1, 1)
    elif channels > 3:
        image = image[:3]

    return image.permute(1, 2, 0).numpy()


def normalize_array(array, eps=1e-8):
    array = np.asarray(array, dtype=np.float32)
    return (array - array.min()) / (array.max() - array.min() + eps)


def feature_activation_grid(activation, n_tiles=16, tile_size=72, cols=4, gap=4):
    features = activation.detach().float().cpu()
    if features.ndim == 4:
        features = features[0]

    channels = features.shape[0]
    n_tiles = min(n_tiles, channels)
    rows = math.ceil(n_tiles / cols)

    scores = features.flatten(1).std(dim=1)
    selected = torch.topk(scores, k=n_tiles).indices.tolist()

    grid_h = rows * tile_size + (rows - 1) * gap
    grid_w = cols * tile_size + (cols - 1) * gap
    grid = np.zeros((grid_h, grid_w), dtype=np.float32)

    for tile_idx, channel_idx in enumerate(selected):
        row = tile_idx // cols
        col = tile_idx % cols
        fmap = features[channel_idx].unsqueeze(0).unsqueeze(0)
        fmap = F.interpolate(fmap, size=(tile_size, tile_size), mode="bilinear", align_corners=False)
        fmap = normalize_array(fmap.squeeze().numpy())
        y0 = row * (tile_size + gap)
        x0 = col * (tile_size + gap)
        grid[y0:y0 + tile_size, x0:x0 + tile_size] = fmap

    return grid


def classifier_probability_grid(logits, tile_size=100, gap=4):
    logits = logits.detach().float().cpu()
    if logits.ndim == 4:
        logits = logits[0]

    if logits.shape[0] == 1:
        probs = torch.sigmoid(logits)
    else:
        probs = torch.softmax(logits, dim=0)

    n_tiles = min(probs.shape[0], 4)
    rows = math.ceil(n_tiles / 2)
    cols = min(n_tiles, 2)
    grid_h = rows * tile_size + (rows - 1) * gap
    grid_w = cols * tile_size + (cols - 1) * gap
    grid = np.zeros((grid_h, grid_w), dtype=np.float32)

    for tile_idx in range(n_tiles):
        row = tile_idx // cols
        col = tile_idx % cols
        prob = probs[tile_idx].unsqueeze(0).unsqueeze(0)
        prob = F.interpolate(prob, size=(tile_size, tile_size), mode="bilinear", align_corners=False)
        prob = normalize_array(prob.squeeze().numpy())
        y0 = row * (tile_size + gap)
        x0 = col * (tile_size + gap)
        grid[y0:y0 + tile_size, x0:x0 + tile_size] = prob

    return grid


def output_prediction_grid(logits, tile_size=120, gap=6):
    logits = logits.detach().float().cpu()
    if logits.ndim == 4:
        logits = logits[0]

    if logits.shape[0] == 1:
        probability = torch.sigmoid(logits[0])
        prediction = (probability > 0.5).float()
    else:
        probability = torch.softmax(logits, dim=0).max(dim=0).values
        prediction = torch.argmax(logits, dim=0).float()

    maps = [probability, prediction]
    grid_h = tile_size
    grid_w = 2 * tile_size + gap
    grid = np.zeros((grid_h, grid_w), dtype=np.float32)

    for i, map_tensor in enumerate(maps):
        tile = map_tensor.unsqueeze(0).unsqueeze(0)
        tile = F.interpolate(tile, size=(tile_size, tile_size), mode="nearest")
        tile = normalize_array(tile.squeeze().numpy())
        x0 = i * (tile_size + gap)
        grid[:, x0:x0 + tile_size] = tile

    return grid

In [ ]:
def extract_multilevel_features(model, image_batch, feature_layers):
    activations = {}
    handles = []

    def capture(name):
        def hook(_module, _inputs, output):
            activations[name] = output.detach()
        return hook

    for name in ["low", "mid", "high"]:
        handles.append(feature_layers[name].register_forward_hook(capture(name)))

    try:
        with torch.no_grad():
            logits = model(image_batch)
    finally:
        for handle in handles:
            handle.remove()

    return activations, logits


def draw_pipeline_box(ax, x, y, w, h, label, edge_color="red"):
    rect = patches.Rectangle((x, y), w, h, fill=False, linewidth=2.2, edgecolor=edge_color)
    ax.add_patch(rect)
    ax.text(
        x + w / 2,
        y + h / 2,
        label,
        ha="center",
        va="center",
        color="white",
        fontsize=11,
        weight="bold" if "Output" in label else "normal",
    )


def draw_arrow(ax, x0, y0, x1, y1, color="red"):
    ax.annotate(
        "",
        xy=(x1, y1),
        xytext=(x0, y0),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, shrinkA=0, shrinkB=0),
    )


def make_feature_extraction_figure(
    model,
    model_name,
    dataloaders,
    feature_layers,
    device,
    custom_stats=None,
    save_dir="figures/feature_extraction",
    save=True,
):
    image_batch, _target = first_available_sample(dataloaders, device)
    activations, logits = extract_multilevel_features(model, image_batch, feature_layers)

    input_image = tensor_to_display_image(image_batch, custom_stats=custom_stats)
    low_grid = feature_activation_grid(activations["low"], n_tiles=20, tile_size=64, cols=5)
    mid_grid = feature_activation_grid(activations["mid"], n_tiles=16, tile_size=68, cols=4)
    high_grid = feature_activation_grid(activations["high"], n_tiles=12, tile_size=72, cols=4)
    classifier_grid = classifier_probability_grid(logits, tile_size=100)
    output_grid = output_prediction_grid(logits, tile_size=120)

    bg_color = "#149bd3"
    fig = plt.figure(figsize=(14.5, 6.0), facecolor=bg_color)
    canvas = fig.add_axes([0, 0, 1, 1], facecolor=bg_color)
    canvas.set_xlim(0, 1)
    canvas.set_ylim(0, 1)
    canvas.axis("off")

    image_ax = fig.add_axes([0.025, 0.70, 0.17, 0.23], facecolor=bg_color)
    image_ax.imshow(input_image)
    image_ax.axis("off")

    top_y = 0.775
    box_h = 0.12
    boxes = [
        (0.235, top_y, 0.105, box_h, "Low Level\nFeatures"),
        (0.405, top_y, 0.105, box_h, "Mid Level\nFeatures"),
        (0.575, top_y - 0.025, 0.105, box_h + 0.05, "High\nLevel\nFeatures"),
        (0.735, top_y, 0.115, box_h, "Trainable\nClassifier"),
        (0.860, top_y - 0.02, 0.13, box_h + 0.04, "Output\n(predicted\nmask)"),
    ]

    for x, y, w, h, label in boxes:
        draw_pipeline_box(canvas, x, y, w, h, label)

    arrow_y = top_y + box_h / 2
    draw_arrow(canvas, 0.195, arrow_y, boxes[0][0], arrow_y)
    for left, right in zip(boxes[:-1], boxes[1:]):
        x0 = left[0] + left[2]
        x1 = right[0]
        y0 = left[1] + left[3] / 2
        y1 = right[1] + right[3] / 2
        draw_arrow(canvas, x0, y0, x1, y1)

    grid_axes = [
        fig.add_axes([0.135, 0.08, 0.18, 0.52], facecolor=bg_color),
        fig.add_axes([0.330, 0.08, 0.18, 0.52], facecolor=bg_color),
        fig.add_axes([0.525, 0.08, 0.18, 0.52], facecolor=bg_color),
        fig.add_axes([0.730, 0.17, 0.12, 0.36], facecolor=bg_color),
        fig.add_axes([0.855, 0.17, 0.13, 0.36], facecolor=bg_color),
    ]
    grids = [low_grid, mid_grid, high_grid, classifier_grid, output_grid]

    for ax, grid in zip(grid_axes, grids):
        ax.imshow(grid, cmap="gray", vmin=0, vmax=1)
        ax.axis("off")

    canvas.text(
        0.02,
        0.04,
        model_name,
        ha="left",
        va="bottom",
        color="white",
        fontsize=9,
        alpha=0.9,
    )

    if save:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        safe_name = model_name.replace("/", "_").replace(" ", "_")
        save_path = Path(save_dir) / f"{safe_name}_multilevel_feature_extraction.png"
        fig.savefig(save_path, dpi=220, bbox_inches="tight", facecolor=fig.get_facecolor())
        print(f"Saved: {save_path}")

    return fig

In [ ]:
fig = make_feature_extraction_figure(
    model=model_imagenet,
    model_name=model_name_imagenet,
    dataloaders=fallback_dataloaders[0],
    feature_layers=feature_layers_to_compare[0],
    device=device,
    custom_stats=custom_stats_list[0],
    save_dir="figures/feature_extraction",
    save=True,
)
plt.show()

In [ ]:
for model, model_name, dataloaders, feature_layers, custom_stats in zip(
    models_to_compare,
    names_to_compare,
    fallback_dataloaders,
    feature_layers_to_compare,
    custom_stats_list,
):
    fig = make_feature_extraction_figure(
        model=model,
        model_name=model_name,
        dataloaders=dataloaders,
        feature_layers=feature_layers,
        device=device,
        custom_stats=custom_stats,
        save_dir="figures/feature_extraction/all_models",
        save=True,
    )
    plt.show()